# KAF-3 — Consumer groups & rebalancing

**Break → Detect → Fix → Prove.** Consumers in the same **group** split a topic's partitions
among themselves — each partition has exactly one owner. When membership changes (a consumer
joins, leaves, or is *presumed dead*), the group **rebalances**: partitions are revoked and
reassigned. During a stop-the-world rebalance every consumer **pauses**, and uncommitted work on a
revoked partition gets **reprocessed or lost**.

**Pre-requisite:** the Kafka broker is up (`make up`). Consumers here run on the **host** listener
`localhost:29092`; inspect the group live in **kafka-ui** at http://localhost:8080 → topic →
**Consumers**.

**Laptop-safe:** a bounded ~600-message batch, no infinite loops — every consumer uses
`consumer_timeout_ms` so polls return on their own and is `close()`d; the topic is deleted at
teardown.

See the companion writeup in [`README.md`](./README.md).

## What runs for real vs. what we describe

True multi-process rebalancing — a separate worker dying mid-batch, the wall-clock **pause** of a
stop-the-world rebalance, a `session.timeout.ms` expiry firing in real time — needs more than one
OS process and **can't be faithfully reproduced in one notebook**. Like the SPK-2 / SPK-3 OOM
modules, this one is explicit about the boundary:

- **Demonstrated for real (below):** several `KafkaConsumer`s sharing one `group_id` form a group
  *in this process*; we watch `.assignment()` **split** as a member joins and **merge back** as
  one leaves. That reassignment *is* the rebalance — the observable core.
- **Described with snippets (not executed):** the stop-the-world *pause*, eviction of a slow
  consumer on `max.poll.interval.ms`, and the multi-process config fixes (`session.timeout.ms`,
  static membership, cooperative-sticky) — shown as the code/config you'd use in a real deploy.

In [ ]:
from common.kafka_helpers import ensure_topic, produce_events, delete_topic, BOOTSTRAP
from kafka import KafkaConsumer

TOPIC = "kaf3_orders"
GROUP = "kaf3-grp"
PARTITIONS = 3

# consumers below all use a short timeout so .poll()/iteration always returns (no hangs).
POLL_MS = 4000

def new_consumer(client_id):
    # A group member. Same GROUP as the others -> they share TOPIC's partitions.
    # consumer_timeout_ms makes blocking calls return on their own (laptop-safe).
    return KafkaConsumer(
        TOPIC,
        bootstrap_servers=BOOTSTRAP,          # host / external listener
        group_id=GROUP,                        # SAME group -> one group, shared partitions
        client_id=client_id,
        auto_offset_reset="earliest",
        enable_auto_commit=False,              # commit explicitly (see KAF-2 / step 5)
        consumer_timeout_ms=POLL_MS,
    )

def assigned(c):
    # Partitions currently owned by consumer c, as a sorted list of ints.
    return sorted(tp.partition for tp in c.assignment())

print("producers/consumers ->", BOOTSTRAP, "| kafka-ui:", "http://localhost:8080")

## Step 0 — a 3-partition topic + a bounded batch

Three partitions means the group can have up to 3 active members (one per partition). We produce a
small bounded batch so each consumer has something to poll and trigger its assignment.

In [ ]:
ensure_topic(TOPIC, num_partitions=PARTITIONS, recreate=True)
n = produce_events(TOPIC, 600, key_fn=lambda i: f"cust-{i % 200}")  # spread across partitions
print(f"created {TOPIC} with {PARTITIONS} partitions, produced {n} events")

## 1. Break it — assignment shifts as the group changes

We change group membership three times and print who owns what after each change. A `poll()` is
what makes a consumer actually **join** and get its assignment, so we poll after every change.

### Step 1 — consumer A alone → owns all 3 partitions

A is the only member, so the group assigns it **every** partition.

In [ ]:
A = new_consumer("A")
A.poll(timeout_ms=POLL_MS)     # first poll triggers the join + assignment
print("A assignment:", assigned(A), "  (expect all 3: [0, 1, 2])")
step1 = {"A": assigned(A), "B": []}

### Step 2 — consumer B joins the same group → assignment **rebalances** (split)

Starting B with the **same** `group_id` is a membership change → the broker rebalances the group.
We poll **both** consumers so each re-joins and settles on its new share (~A:2 / B:1). This split
*is* the rebalance.

In [ ]:
B = new_consumer("B")
# poll both a few times: the join is multi-round (revoke -> rejoin -> new assignment).
for _ in range(3):
    A.poll(timeout_ms=POLL_MS)
    B.poll(timeout_ms=POLL_MS)
print("A assignment:", assigned(A))
print("B assignment:", assigned(B))
print("union covers all partitions:", sorted(set(assigned(A)) | set(assigned(B))))
step2 = {"A": assigned(A), "B": assigned(B)}

### Step 3 — "kill" B (`B.close()`) → A reclaims all 3 (rebalance back)

`B.close()` sends a graceful `LeaveGroup` — our in-process stand-in for a consumer crash/scale-down.
That's another membership change, so the group rebalances again and A, now the only member,
**reclaims all 3** partitions.

> Hazard this models: if B held uncommitted offsets when its partition was revoked, A re-reads them
> → **duplicate processing** (or **lost** work if auto-commit had committed ahead of processing).
> Rebalances are exactly when the commit discipline from KAF-2 matters most.

In [ ]:
B.close()                       # B leaves the group -> triggers a rebalance
for _ in range(3):
    A.poll(timeout_ms=POLL_MS)  # A rejoins and picks up B's old partitions
print("A assignment:", assigned(A), "  (expect all 3 again: [0, 1, 2])")
step3 = {"A": assigned(A), "B": []}

## 2. Detect it

- **`consumer.assignment()`** after a poll = the partitions that member owns; it **changed** at
  each join/leave above. That change *is* the rebalance.
- **kafka-ui** (http://localhost:8080 → topic `kaf3_orders` → **Consumers** → `kaf3-grp`) shows the
  members and their partitions live — the member count and ownership shift as the group changes.
- In a real deployment, consumer **logs** show `Revoking previously assigned partitions ...` /
  `Setting newly assigned partitions ...` — one pair per rebalance; many back-to-back = a *storm*.

## 3. Diagnose — why rebalances happen and why they hurt

A rebalance fires on any **membership change**: a consumer **joins** (deploy/scale-up), **leaves**
gracefully (`close()`), is **presumed dead** (misses heartbeats for `session.timeout.ms`, *or*
exceeds `max.poll.interval.ms` between polls — a slow batch *looks* dead though the process is
alive), or the topic's **partition count** changes.

Why it hurts (classic **eager** rebalancing, the default before cooperative): it's
**stop-the-world** — every member revokes *all* its partitions and consumption **pauses across the
whole group** until the new assignment is agreed. More/longer rebalances = more wall-clock time
paused instead of processing, plus reprocessing/loss on any uncommitted revoked partition.

The pause can't be shown faithfully in one process — here's the shape of it in a real consumer:

```python
# Real multi-process consumer: the rebalance pause happens between these callbacks.
from kafka import ConsumerRebalanceListener

class Listener(ConsumerRebalanceListener):
    def on_partitions_revoked(self, revoked):
        consumer.commit()           # commit BEFORE losing the partitions -> avoid duplicates
        # ... consumption is PAUSED from here until on_partitions_assigned fires ...
    def on_partitions_assigned(self, assigned):
        pass                        # resume; processing was stopped for the whole gap

consumer.subscribe([TOPIC], listener=Listener())
```

## 4 & 5. Fix it / mitigate (config — runtime effect is multi-process)

What's observable in one process is the **assignment split/merge** above. The *runtime* payoff of
these fixes (skipped rebalance, incremental rebalance, no false eviction) is a multi-process
property, so we show the **config** you'd set. A consumer built with these is real and valid —
it simply can't exhibit the cross-process timing here.

```python
fixed = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP,
    group_id=GROUP,

    # (a) Liveness tuning — stop a slow-but-alive consumer being falsely evicted (the #1
    #     avoidable cause of rebalance storms). Raise max.poll.interval.ms above worst-case batch.
    session_timeout_ms=45000,
    heartbeat_interval_ms=15000,        # ~1/3 of session_timeout_ms
    max_poll_interval_ms=300000,        # > your slowest batch's processing time
    max_poll_records=500,               # bound work per poll so you keep polling

    # (b) Static membership — a restart with the SAME id rejoins as the same member within
    #     session.timeout.ms, so the broker SKIPS the rebalance entirely (kills deploy churn).
    group_instance_id="worker-1",       # stable, unique per instance

    # (c) Cooperative-sticky assignor — INCREMENTAL rebalances: only partitions that actually
    #     move are revoked; everyone else keeps consuming (no stop-the-world).
    partition_assignment_strategy=["org.apache.kafka.clients.consumer.CooperativeStickyAssignor"],

    enable_auto_commit=False,           # commit AFTER processing (KAF-2) so a revoke can't lose work
    consumer_timeout_ms=POLL_MS,
)
```

Plus a sizing rule that needs no code: **#consumers ≤ #partitions** — extra consumers own no
partition (idle) yet still rebalance. With 3 partitions, a 4th member just sits empty:

In [ ]:
# Demonstrate the sizing limit: a 4th member can't own a partition (only 3 exist).
A.poll(timeout_ms=POLL_MS)
C = new_consumer("C"); D = new_consumer("D"); E = new_consumer("E")
for _ in range(4):
    for c in (A, C, D, E):
        c.poll(timeout_ms=POLL_MS)
owners = {"A": assigned(A), "C": assigned(C), "D": assigned(D), "E": assigned(E)}
idle = [name for name, parts in owners.items() if not parts]
print("4 members, 3 partitions ->", owners)
print("idle member(s) owning nothing:", idle, " (extra consumers beyond #partitions sit idle)")
for c in (C, D, E):
    c.close()   # drop the extras; rebalances A back toward all 3

## 6. Prove it — the assignment table (3 → split → 3)

Ownership moved when B joined and returned when B left; the partition count is conserved at 3, and
each partition always has exactly one owner. That *split → merge* is the rebalance, proven.

In [ ]:
def cover(step):
    return sorted(set(step["A"]) | set(step["B"]))
rows = [
    ("1. A alone",            step1, len(cover(step1))),
    ("2. B joins (rebalance)",step2, len(cover(step2))),
    ("3. B leaves (rebal.)",  step3, len(cover(step3))),
]
print(f"{'step':<26}{'A owns':<12}{'B owns':<12}{'partitions covered':>18}")
for label, s, n_cov in rows:
    print(f"{label:<26}{str(s['A']):<12}{str(s['B'] or '-'):<12}{n_cov:>18}")
print("\nConserved at 3, split->merge = the rebalance.")

## Takeaways & "in real production…"

- **Rebalances are costly — minimize them.** Every membership change pauses the group (fully, with
  eager assignment); frequent rebalances ("storms") are an availability problem.
- **#consumers ≤ #partitions.** Partitions cap useful parallelism; extra consumers idle but still
  rebalance. Add partitions first if you need more throughput.
- **Most avoidable rebalances are false-death evictions.** Tune `session.timeout.ms` /
  `heartbeat.interval.ms` and set `max.poll.interval.ms` above your worst-case batch.
- **Static membership** (`group.instance.id`) so rolling restarts don't rebalance; **cooperative-
  sticky** so the rebalances you can't avoid are incremental, not stop-the-world.
- **Pair with offset discipline (KAF-2):** commit after processing + idempotent sink, so the
  reprocessing a revoke causes is harmless.

## Teardown

Close every consumer (no lingering group members) and delete the topic. `make clean` also clears
any local Kafka / `.tmp/` state.

In [ ]:
for c in (A, B):
    try:
        c.close()
    except Exception:
        pass
delete_topic(TOPIC)
print("Closed all consumers; deleted", TOPIC, "(`make clean` clears any local state).")